# 00 · Download the five external sources

Run once on your own computer (the sandbox used for the analysis has no access to these hosts).
Every file is recorded in `data/external/manifest.json` with URL, download date and SHA-256, so the report can cite the exact version.

| Source | Publisher | Licence | Used for |
|---|---|---|---|
| OpenStreetMap POIs (Overpass API) | OSM contributors | ODbL | restaurants, nightlife, culture, attractions, hotels, parks, daily shops |
| DVF géolocalisées 2022–2025, Paris | DGFiP / Etalab | Licence Ouverte 2.0 | apartment sale price per m² around each listing |
| IRIS boundaries, Paris | IGN / INSEE (Opendatasoft copy) | Licence Ouverte | point-in-polygon join to IRIS districts |
| Filosofi 2021 by IRIS | INSEE | Licence Ouverte | household income, poverty rate, inequality |
| Recorded crime by commune | SSMSI (Ministry of the Interior) | Licence Ouverte | crime rates per 1,000 residents by arrondissement |
| IDFM stations | Île-de-France Mobilités | Licence ODbL | already in the classmate's repo (`data/stations_used.csv`) |

This notebook is **not executed** in the repository: its output depends on the day it is run.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import urllib.request, urllib.parse, hashlib, json, gzip, re, time, datetime
from config import EXT as OUT
OUT.mkdir(parents=True, exist_ok=True)
UA = {"User-Agent": "dauphine-ds-project/0.1 (student project)", "Accept": "*/*"}
MANIFEST = json.loads((OUT / "manifest.json").read_text()) if (OUT / "manifest.json").exists() else {}


def record(url, name):
    MANIFEST[name] = {"url": url, "bytes": (OUT / name).stat().st_size,
                      "sha256": hashlib.sha256((OUT / name).read_bytes()).hexdigest(), "downloaded": datetime.date.today().isoformat()}


def get(url, name, data=None, tries=1, wait=20):
    """Download once; skip if the file is already there. Returns True on success."""
    if (OUT / name).exists() and (OUT / name).stat().st_size > 1000:
        print(f"SKIP  {name}"); record(url, name); return True
    for i in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, data=data, headers=UA), timeout=600) as r:
                body = r.read()
            (OUT / name).write_bytes(body); print(f"OK    {name}  {len(body) / 1e6:.1f} MB"); record(url, name); return True
        except Exception as e:
            print(f"FAIL  {name} ({url.split('/')[2]}): {e}")
            if i + 1 < tries: time.sleep(wait * (i + 1))
    return False

## 1. OpenStreetMap points of interest (Paris bounding box)
The main Overpass server is often at capacity; two mirrors are tried in turn.

In [ ]:
Q = """[out:json][timeout:300];
(
  nwr["amenity"~"^(restaurant|cafe|bar|pub|fast_food|theatre|cinema|nightclub)$"](48.80,2.22,48.91,2.47);
  nwr["tourism"~"^(attraction|museum|gallery|viewpoint|hotel)$"](48.80,2.22,48.91,2.47);
  nwr["leisure"="park"](48.80,2.22,48.91,2.47);
  nwr["shop"~"^(supermarket|bakery|convenience|department_store|mall)$"](48.80,2.22,48.91,2.47);
);
out center tags;"""
body = urllib.parse.urlencode({"data": Q}).encode()
for server in ("https://overpass-api.de", "https://overpass.kumi.systems", "https://overpass.private.coffee", "https://overpass-api.de"):
    if get(server + "/api/interpreter", "osm_paris_poi.json", data=body, tries=2, wait=30):
        break

## 2. DVF property transactions, Paris (département 75)

In [ ]:
for year in (2022, 2023, 2024, 2025):
    get(f"https://files.data.gouv.fr/geo-dvf/latest/csv/{year}/departements/75.csv.gz", f"dvf_75_{year}.csv.gz")

## 3. IRIS boundaries and 4. Filosofi 2021 income

In [ ]:
get("https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/georef-france-iris/exports/geojson?where=dep_code%3D%2275%22",
    "iris_paris.geojson")
get("https://www.insee.fr/fr/statistiques/fichier/8229323/BASE_TD_FILO_IRIS_2021_DISP_CSV.zip", "filosofi_iris_2021_disp.zip")
get("https://www.insee.fr/fr/statistiques/fichier/8229323/BASE_TD_FILO_IRIS_2021_DEC_CSV.zip", "filosofi_iris_2021_dec.zip")

## 5. Recorded crime
The national file is 40 MB; only Paris rows are kept. The filter checks the commune-code column only (an earlier version matched crime-rate values such as `1,7511955`).

In [ ]:
CRIME = ("https://static.data.gouv.fr/resources/bases-statistiques-communale-departementale-et-regionale-de-la-delinquance-enregistree-par-la-police-et-la-gendarmerie-nationales/"
         "20260709-115942/donnee-data.gouv-2025-geographie2026-produit-le2026-06-25.csv.gz")
if not (OUT / "crime_paris.csv").exists() and get(CRIME, "crime_communes_france.csv.gz"):
    pat = re.compile(r'^"?75(056|1\d\d)"?[;,]')
    with gzip.open(OUT / "crime_communes_france.csv.gz", "rt", encoding="utf-8", errors="replace") as f, \
            open(OUT / "crime_paris.csv", "w", encoding="utf-8") as g:
        g.write(next(f)); n = 0
        for line in f:
            if pat.match(line): g.write(line); n += 1
    print(f"OK    crime_paris.csv  {n} rows"); record(CRIME, "crime_paris.csv")
    (OUT / "crime_communes_france.csv.gz").unlink()

In [ ]:
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2))
want = ["osm_paris_poi.json", "dvf_75_2024.csv.gz", "iris_paris.geojson", "filosofi_iris_2021_disp.zip", "crime_paris.csv"]
missing = [w for w in want if not (OUT / w).exists()]
print("DONE" if not missing else f"MISSING: {missing} (re-run; finished files are skipped)")